# Label lab: specimen label table and feather IDs

Reads the label fields from the **original** slides, gives every segmented feather its ID, scores everything against
hand-checked answers, and lets you look at each slide.

| Field | Where | Notes |
|---|---|---|
| `bird_id` | large code, top-center box (e.g. `A 0012 H` -> `A0012H`) | backup: leading ID of the file name (`A0012`) |
| `species`, `age`, `sex` | label row below the ID, top-left box, top-right box | the small code next to the species (e.g. `3K+`) is ignored |
| `date` -> `year`, `month`, `day` | blue date, bottom-right | the short code just above it (e.g. `2B`) is ignored |
| `feather_id` | one per segmented feather | `<bird_id>_<year>_<letter>`, letters A, B, C ... left to right, e.g. `A0012H_1996_A` |

`file_id` is the file name (`A0012 1996-im1058.jpg`). The red/blue text line under the label boxes is ignored.

**Statuses**: `readable` · `needs_review` (bad format, low-confidence partly hidden code, or two readings disagree) ·
`conflict` (disagrees with the file name) · `filename_fallback` (nothing on the slide, value from the file name) ·
`not_found` (absent in every crop, exported `N/A`) · `model_error` (request failed or the answer was garbled; **not**
the same as missing, exported blank, retried on the next run).

**Use it**: inspect a finished run (`RUN_NEW = False`, set `RUN_DIR`), or run a new sample (`RUN_NEW = True`).
Labels run from here (`MODE = "local"`, via the VLM server) or on the workers (`MODE = "distributed"`, once the
label code is deployed). Feather IDs need segmentation: `RUN_SEGMENTATION = True` sends the same slides through the
workers' normal pipeline (outputs copied to `data/sample_runs/`), or point `SEGMENTATION_DIR` at an earlier one.
Kernel: **bird-feather-analysis (pixi)**.

## 1. Parameters

In [ ]:
RUN_NEW = False                       # True: run SELECT below; False: inspect RUN_DIR
RUN_DIR = "../data/label_runs/sample30-v4"   # run folder to inspect (or resume with RUN_NEW = True)
SEGMENTATION_DIR = "../data/sample_runs/test-sample30-seg"   # segmentation output for feather IDs (or None)

SELECT = "manifest"                   # "manifest" | "stratified" | "random" | "match"
MANIFEST = "../configs/label_sample_30.txt"
N, SEED = 10, 1                       # for "stratified" / "random"
MATCH = ["A0012 1996"]                # for "match": substrings of file names
MODE = "local"                        # labels: "local" (here + VLM server) | "distributed" (cluster workers)
RUN_SEGMENTATION = True               # with RUN_NEW: also segment the slides on the workers for feather IDs
FIELDS = None                         # e.g. ["date", "bird_id"] to iterate on a few fields; None = all
WORKERS = 3                           # slides in flight at once (local mode)

GRID_ORDER = "worst"                  # slide grid: "worst" first, or "manifest" order
GRID_ONLY_PROBLEMS = False            # True: only slides with something to look at
CONFIG = "../configs/label_fields.json"
GOLD = "../data/label_gold.csv"       # hand-checked answers (blank = not scored, N/A = truly absent)

## 2. Setup

In [ ]:
import base64, html, io, json, random, sys, time
from pathlib import Path
from urllib.parse import quote
sys.path.insert(0, "..")
import pandas as pd
from IPython.display import display, HTML, Markdown
from PIL import Image, ImageDraw
from src import label_extract as lx
from src.notebook_remote_runtime import RemoteClusterIO, load_notebook_config_from_env

pd.set_option("display.max_colwidth", 80)
rc = load_notebook_config_from_env()
remote = RemoteClusterIO(rc)
CACHE = Path("../data/raw_cache"); CACHE.mkdir(parents=True, exist_ok=True)

def local_copy(path_or_name):
    """The slide in data/raw_cache, fetched from the head node if needed."""
    p = CACHE / Path(path_or_name).name
    if not p.exists():
        remote.fetch_remote_file(rc.head_host, remote.remote_runtime_path(f"{rc.remote_input_dir}/{p.name}"), CACHE)
    return p

## 3. Run a new sample (only when `RUN_NEW = True`)
Labels and segmentation run one after the other; both can be resumed by running this cell again.

In [ ]:
if RUN_NEW:
    cfg = lx.load_config(CONFIG)
    all_images = remote.list_remote_images()
    by_name = {Path(p).name: p for p in all_images}
    if SELECT == "manifest":
        names = [l.strip() for l in Path(MANIFEST).read_text().splitlines() if l.strip() and not l.startswith("#")]
        chosen = [by_name[n] for n in names if n in by_name]
    elif SELECT == "stratified":
        chosen = lx.stratified_sample(all_images, N, SEED)
    elif SELECT == "match":
        chosen = [p for p in all_images if any(m in Path(p).name for m in MATCH)]
    else:
        chosen = random.Random(SEED).sample(all_images, min(N, len(all_images)))
    out = Path(RUN_DIR) if RUN_DIR and Path(RUN_DIR).exists() else Path("../data/label_runs") / time.strftime("%Y%m%d-%H%M%S")
    out.mkdir(parents=True, exist_ok=True)
    (out / "config_used.json").write_text(json.dumps(cfg, indent=2))
    (out / "manifest.txt").write_text("\n".join(Path(p).name for p in chosen) + "\n")
    todo = lx.pending_images(chosen, out)
    print(f"labels: {len(chosen)} slides -> {out}  ({len(chosen) - len(todo)} done, {len(todo)} to run, mode={MODE})")
    t0 = time.time()
    if MODE == "distributed":
        lx.run_distributed(todo, out, log=lambda s: print(s, flush=True))
    else:
        lx.run_local(todo, out, cfg, FIELDS, WORKERS, log=lambda s: print(s, flush=True))
    print(f"labels done in {time.time() - t0:.0f}s")
    RUN_DIR = str(out)
    if RUN_SEGMENTATION:
        from src import sample_run
        seg_id = f"test-{out.name}-seg"
        SEGMENTATION_DIR = str(sample_run.run(match=[Path(p).name for p in chosen], timeout=5400, run_id=seg_id))
else:
    print("RUN_NEW = False: inspecting", RUN_DIR, "| segmentation:", SEGMENTATION_DIR)

## 4. Load the run

In [ ]:
run = Path(RUN_DIR)
cfg = json.loads((run / "config_used.json").read_text()) if (run / "config_used.json").exists() else lx.load_config(CONFIG)
latest = lx.load_previous(run / "labels.jsonl")
order = ([l.strip() for l in (run / "manifest.txt").read_text().splitlines() if l.strip() and not l.startswith("#")]
         if (run / "manifest.txt").exists() else list(latest))
summary = lx.finish_run(run, order, cfg, GOLD, SEGMENTATION_DIR)       # refresh exports, feather IDs, scores
latest = lx.load_previous(run / "labels.jsonl")
results = [latest[n] for n in order if n in latest]
if SEGMENTATION_DIR and Path(SEGMENTATION_DIR).exists():
    lx.attach_feathers_from_crops(results, SEGMENTATION_DIR)
gold = lx.load_gold(GOLD) if Path(GOLD).exists() else {}
problems = {r.file_id: lx.slide_problems(r, gold.get(r.file_id), cfg) for r in results}
print(f"{run}: {summary['slides']} slides, {summary['needs_review']} need review, "
      f"{summary['slides_with_model_error']} with model errors, {summary['mean_seconds_per_slide']} s/slide, "
      f"feather IDs for {sum(bool(r.feathers) for r in results)} slides")
display(pd.DataFrame(summary["status_counts"]).fillna(0).astype(int).T)

## 5. Scores against the hand-checked answers
`wrong_value` = a wrong value accepted as readable (the costly kind); `false_na` = the field was there but the run said not found. Feather count = number of segmented feathers vs the number on the slide (the pipeline keeps at most 5).

In [ ]:
if "gold" in summary:
    exact = pd.DataFrame({k: {kk: v[kk] for kk in ("n", "correct", "accuracy", "wrong_value", "false_na")}
                          for k, v in summary["gold"].items()}).T
    display(exact)
    fc = summary.get("gold_feather_count")
    print(f"feather count: {fc['correct']}/{fc['n']} slides right" if fc else "feather count: no segmentation output yet")
else:
    print("no gold file at", GOLD)

## 6. Slide grid
One card per slide. **Click the image** to open the full-resolution slide in a new JupyterLab tab (zoom with the
image viewer). Border: red = a wrong or missed value / model error / wrong feather count, amber = needs a look
(review, conflict, file-name fallback), green = clean. Field chips use the status colours; feather crops below
link to the full crop.

In [ ]:
STATUS_COLORS = {"readable": "#d4edda", "partial": "#fff3cd", "not_found": "#e2e3e5", "filename_fallback": "#fff3cd",
                 "needs_review": "#f8d7da", "conflict": "#f8d7da", "model_error": "#f5c6cb"}
SEVERE = {"wrong", "missed", "model_error", "count"}

def _thumb(path, size):
    im = Image.open(path).convert("RGB"); im.thumbnail(size)
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=80)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def _rel(path):
    """Link that JupyterLab opens in a new tab (relative to this notebook)."""
    return quote(str(Path("..") / Path(path).resolve().relative_to(Path("..").resolve())))

def _card(r):
    probs = problems[r.file_id]
    border = "#c0392b" if any(p["kind"] in SEVERE for p in probs) else ("#e0a800" if probs else "#2e8b57")
    slide = local_copy(r.image)
    chips = "".join(
        f'<span style="background:{STATUS_COLORS.get(f.status, "#eee")};padding:1px 5px;margin:1px;border-radius:3px;'
        f'display:inline-block" title="{html.escape(f.status + (": " + f.note if f.note else ""))}">'
        f'<b>{n}</b> {html.escape(lx.export_value(f.value, f.status) or "-")}</span>' for n, f in r.fields.items())
    plist = "".join(
        f'<li style="color:{"#c0392b" if p["kind"] in SEVERE else "#8a6d00"}"><b>{p["field"]}</b> {p["kind"]}: '
        f'got {html.escape(str(p.get("got")))}' + (f', want {html.escape(str(p["want"]))}' if p.get("want") else "")
        + (f' <i>({html.escape(p["note"])})</i>' if p.get("note") else "") + "</li>" for p in probs)
    feathers = "".join(
        f'<a href="{_rel(Path(SEGMENTATION_DIR) / f["crop"])}" target="_blank" title="{f["feather_id"]}">'
        f'<img src="{_thumb(Path(SEGMENTATION_DIR) / f["crop"], (60, 140))}" style="height:70px;margin:1px"></a>'
        for f in r.feathers if SEGMENTATION_DIR and (Path(SEGMENTATION_DIR) / f["crop"]).exists())
    ids = ", ".join(f["feather_id"] for f in r.feathers) or "<i>no segmentation yet</i>"
    return (f'<div style="border:4px solid {border};border-radius:6px;padding:6px;width:300px;font-size:12px;'
            f'font-family:sans-serif;vertical-align:top">'
            f'<a href="{_rel(slide)}" target="_blank"><img src="{_thumb(slide, (288, 330))}" style="width:288px"></a>'
            f'<div style="font-weight:bold;margin-top:4px">{html.escape(r.file_id)}</div>'
            f'<div>{chips}</div><div style="margin-top:3px"><b>feathers</b> {ids}</div><div>{feathers}</div>'
            f'<ul style="margin:3px 0 0 16px;padding:0">{plist}</ul></div>')

def _severity(r):
    probs = problems[r.file_id]
    return (-sum(p["kind"] in SEVERE for p in probs), -len(probs))

cards = [r for r in results if problems[r.file_id] or not GRID_ONLY_PROBLEMS]
if GRID_ORDER == "worst":
    cards.sort(key=_severity)
n_bad = sum(any(p["kind"] in SEVERE for p in problems[r.file_id]) for r in results)
n_look = sum(bool(problems[r.file_id]) for r in results) - n_bad
display(Markdown(f"**{len(results)} slides**: {n_bad} red, {n_look} amber, {len(results) - n_bad - n_look} green"))
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px">' + "".join(_card(r) for r in cards) + "</div>"))

## 7. Feather IDs

In [ ]:
fdf = pd.read_csv(run / "feathers.csv") if (run / "feathers.csv").exists() else pd.DataFrame()
if len(fdf):
    display(fdf[["file_id", "seq", "feather_id", "bird_id_source", "bird_id_status", "year_source", "year_status"]])
else:
    print("no feather IDs yet: run segmentation (RUN_SEGMENTATION) or set SEGMENTATION_DIR")

## 8. Label table

In [ ]:
df = pd.DataFrame([r.row() for r in results])
value_cols = ["file_id", "bird_id", "species", "age", "sex", "year", "month", "day", "feather_count", "filename_id", "filename_year"]
def _color(col):
    st = df.get(("date" if col in ("year", "month", "day") else col) + "_status")
    return [f"background-color: {STATUS_COLORS.get(s, '')}" for s in st] if st is not None else [""] * len(df)
display(df[[c for c in value_cols if c in df]].style.apply(lambda c: _color(c.name)))

## 9. Crop review
Every flagged field with the crops the model actually saw (for tuning `configs/label_fields.json`).

In [ ]:
REVIEW = ("needs_review", "conflict", "filename_fallback", "model_error")
def evidence_images(r, field):
    paths = [Path(a.evidence) for a in r.fields[field].attempts if a.evidence]
    local = [p for p in paths if p.exists()]
    if local:
        return [(p.stem, Image.open(p)) for p in local]
    im = Image.open(local_copy(r.image))
    return [(f"{field}_{name}", lx.crop_field(im, reg, 900)) for name, reg in lx.crop_plan(cfg["fields"][field], cfg)]

for r in results:
    bad = {n: f for n, f in r.fields.items() if f.status in REVIEW}
    if not bad:
        continue
    display(Markdown(f"---\n### {r.file_id}"))
    for n, f in bad.items():
        print(f"{n}: {f.value!r}  [{f.status}] via {f.source or '-'}  {f.note}")
        for name, im in evidence_images(r, n)[:4]:
            im = im.copy(); im.thumbnail((640, 200)); print("  ", name); display(im)

## 10. One slide, all regions
Every field's crop plan drawn on a slide (thick = primary crop, thin = expanded/fallback crops).

In [ ]:
SLIDE = results[0].file_id if results else "A0012 1996-im1058.jpg"   # any file_id from the table
r = latest.get(SLIDE)
im = Image.open(local_copy(SLIDE)).convert("RGB"); w, h = im.size
d = ImageDraw.Draw(im)
colors = {"bird_id": "red", "species": "orange", "age": "purple", "sex": "blue", "date": "magenta"}
for n, spec in cfg["fields"].items():
    for i, (name, (l, t, rr, b)) in enumerate(lx.crop_plan(spec, cfg)):
        d.rectangle([l*w, t*h, rr*w, b*h], outline=colors.get(n, "red"), width=10 if i == 0 else 3)
    l, t, _, _ = spec["region"]; d.text((l*w + 14, t*h + 14), n, fill=colors.get(n, "red"))
im.thumbnail((900, 900)); display(im)
if r:
    display(pd.DataFrame([{"field": n, "value": f.value, "status": f.status, "source": f.source, "note": f.note}
                          for n, f in r.fields.items()]))

## 11. Outputs
`labels.csv` (one row per slide, `N/A` only for not found), `feathers.csv` (one row per feather ID), `review.csv`, `labels.jsonl` (everything, including every crop attempt and raw model answer), `labels.sqlite3` (`specimen_labels`, `specimen_feathers`), `summary.json`, `evidence/`.

CLI: `pixi run labels -- --manifest configs/label_sample_30.txt --gold data/label_gold.csv --segmentation-dir data/sample_runs/<id>` (`--distributed` for the workers, `--out <dir>` to resume).

In [ ]:
for p in sorted(run.iterdir()):
    print(f"{p.name:22} {'dir' if p.is_dir() else f'{p.stat().st_size/1024:.0f} KB'}")